In [1]:
def main(datasources, start_date, end_date):
    """BigAlpha 提交推理入口: 加载已训练 PatchMixer JSON 权重, 输出 date/instrument/score。

    提交包需要同时包含本 notebook 和 `patchmixer_model.json`。
    平台只会替换 datasources/start_date/end_date；本函数不使用测试区间训练，避免泄漏。
    """
    import os
    import json
    import time
    import numpy as np
    import pandas as pd
    import dai
    import torch
    import torch.nn as nn
    from torch import Tensor
    import structlog

    logger = structlog.get_logger()

    # ---------- 配置: 必须与训练 checkpoint 一致 ----------
    MODEL_FILE = "alstm_model.json"
    INFER_TABLE = datasources["bar1m"]
    SEQ_LEN = 240
    PRED_BATCH = 4096
    INFER_CHUNK_DAYS = 5
    WARMUP_DAYS = 20  # 时序模型推理需向前多取历史, 仅用于构造窗口, 最终输出仍截在 start_date~end_date

    PRICE_COLS = [
        "open",
        "high",
        "low",
        "close",
    ]
    VOL_COLS = []
    FEATURE_COLS = PRICE_COLS + VOL_COLS
    N_FEAT = len(FEATURE_COLS)

    def resolve_model_path():
        candidates = [
            MODEL_FILE,
            os.path.join(os.getcwd(), MODEL_FILE),
        ]
        if "__file__" in globals():
            candidates.append(os.path.join(os.path.dirname(os.path.abspath(__file__)), MODEL_FILE))
        candidates.append(os.path.join("/data/BigAlpha", MODEL_FILE))  # 本地调试兜底
        for p in candidates:
            if p and os.path.exists(p):
                return p
        raise FileNotFoundError(
            f"未找到 {MODEL_FILE}; 提交时请把该 JSON 权重文件与 transformer.ipynb 一并上传")

    # ---------------------------------------------------
    # StockALSTM模型（与训练端定义保持一致）
    # ---------------------------------------------------
    class StockALSTM(nn.Module):
        """带时间注意力机制的 ALSTM，用最近 SEQ_LEN 根 bar 预测下一交易日收益。

        输入:
            x: [B, L, F]
        输出:
            score: [B]
        """

        def __init__(
            self,
            n_feat: int,
            hidden_size: int = 128,
            num_layers: int = 2,
            dropout: float = 0.1,
            rnn_type: str = "GRU",
        ):
            super().__init__()

            if hidden_size < 2:
                raise ValueError("hidden_size 必须大于等于 2")

            try:
                rnn_class = getattr(nn, rnn_type.upper())
            except AttributeError as exc:
                raise ValueError(f"不支持的 rnn_type：{rnn_type}") from exc

            self.n_feat = int(n_feat)
            self.hidden_size = int(hidden_size)
            self.rnn_type = str(rnn_type).upper()

            # 与标准 ALSTM 一致：先将每个时间步映射到隐藏空间。
            self.input_norm = nn.LayerNorm(self.n_feat)
            self.fc_in = nn.Linear(self.n_feat, self.hidden_size)
            self.act = nn.Tanh()

            self.rnn = rnn_class(
                input_size=self.hidden_size,
                hidden_size=self.hidden_size,
                num_layers=num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0.0,
            )

            # 对所有时间步生成归一化注意力权重。
            self.att_net = nn.Sequential(
                nn.Linear(self.hidden_size, self.hidden_size // 2),
                nn.Dropout(dropout),
                nn.Tanh(),
                nn.Linear(self.hidden_size // 2, 1, bias=False),
                nn.Softmax(dim=1),
            )

            # 拼接最后一个时间步状态与注意力汇聚状态。
            self.fc_out = nn.Linear(self.hidden_size * 2, 1)

        def forward(self, x: Tensor) -> Tensor:
            if x.ndim != 3:
                raise ValueError(
                    f"StockALSTM 期望输入形状 [B, L, F]，实际为 {tuple(x.shape)}"
                )
            if x.size(-1) != self.n_feat:
                raise ValueError(
                    f"输入特征维度不一致：模型={self.n_feat}，输入={x.size(-1)}"
                )

            x = self.input_norm(x)
            x = self.act(self.fc_in(x))

            rnn_out, _ = self.rnn(x)
            attention_score = self.att_net(rnn_out)
            attention_output = torch.sum(rnn_out * attention_score, dim=1)

            combined = torch.cat(
                [rnn_out[:, -1, :], attention_output],
                dim=1,
            )
            return self.fc_out(combined).squeeze(-1)

    def load_json_model(model_path, map_location="cpu"):
        with open(model_path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        sd = {}
        for k, meta in payload["state_dict"].items():
            t = torch.tensor(meta["data"], dtype=getattr(torch, meta["dtype"]))
            sd[k] = t.reshape(meta["shape"]).to(map_location)
        payload["state_dict"] = sd
        return payload

    def pool(sd, ed):
        stk = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                        filters={"date": [sd, ed]}).df()
        stk["date"] = pd.to_datetime(stk["date"]).dt.normalize()
        return stk.drop_duplicates(["date", "instrument"]).reset_index(drop=True)

    def iter_date_chunks(sd, ed, days=INFER_CHUNK_DAYS):
        start_ts = pd.to_datetime(sd)
        end_ts = pd.to_datetime(ed)
        cur = start_ts.normalize()
        while cur <= end_ts:
            chunk_start = max(start_ts, cur)
            chunk_end = min(end_ts, cur + pd.Timedelta(days=days) - pd.Timedelta(seconds=1))
            yield chunk_start.strftime("%Y-%m-%d %H:%M:%S"), chunk_end.strftime("%Y-%m-%d %H:%M:%S")
            cur = cur + pd.Timedelta(days=days)

    def build_infer_dataset(table, sd, ed, instruments, stats):
        """按小日期窗口读取注入表并构造每日收盘打分样本。
        每个 chunk 会向前多读 WARMUP_DAYS, 解决平台“时序因子需向前多取 warmup”的校验。
        """
        mean, std = stats
        all_X, all_keys = [], []
        sql = f"SELECT date, instrument, {', '.join(FEATURE_COLS)} FROM {table} ORDER BY instrument, date"
        t0 = time.time()
        for cs, ce in iter_date_chunks(sd, ed):
            read_start = (pd.to_datetime(cs) - pd.Timedelta(days=WARMUP_DAYS)).strftime("%Y-%m-%d %H:%M:%S")
            df = dai.query(sql, filters={"date": [read_start, ce], "instrument": instruments}).df()
            if df.empty:
                continue
            df["date"] = pd.to_datetime(df["date"])
            for c in VOL_COLS:
                df[c] = np.log1p(df[c].clip(lower=0))
            sd_ts, ed_ts = pd.to_datetime(cs), pd.to_datetime(ce)
            wins, keys = [], []
            for ins, sub in df.groupby("instrument", sort=False):
                if len(sub) == 0:
                    continue
           

                feat_df = sub[FEATURE_COLS].copy()

                feat_df = feat_df.replace(
                    [np.inf, -np.inf],
                    np.nan
                )

                # price字段保持原始尺度
                for c in PRICE_COLS:
                    if c in feat_df.columns:
                        feat_df[c] = pd.to_numeric(
                            feat_df[c],
                            errors="coerce"
                        )

                # volume/amount 与训练一致
                for c in VOL_COLS:
                    if c in feat_df.columns:
                        feat_df[c] = np.log1p(
                            pd.to_numeric(
                                feat_df[c],
                                errors="coerce"
                            ).clip(lower=0)
                        )

                # 缺失处理:
                # OHLC: NaN -> 前向填充 -> 0
                # 盘口: 缺失0保持
                feat_df = (
                    feat_df
                    .ffill()
                    .fillna(0.0)
                )
                feats = feat_df.to_numpy(np.float32)
                day = sub["date"].dt.normalize().to_numpy()
                close_pos = np.flatnonzero(np.append(day[1:] != day[:-1], True))
                dates = day[close_pos]
                for p, d0 in zip(close_pos, dates):
                    d = pd.Timestamp(d0)
                    if d < sd_ts.normalize() or d > ed_ts.normalize():
                        continue
                    start = max(0, p - SEQ_LEN + 1)
                    win = feats[start:p + 1]
                    if len(win) < SEQ_LEN:
                        pad = np.repeat(win[:1], SEQ_LEN - len(win), axis=0)
                        win = np.concatenate([pad, win], axis=0)
                    wins.append(win)
                    keys.append((d, ins))
            if wins:
                X = np.stack(wins).astype(np.float32)
                X = ((X - mean) / std).astype(np.float32)
                all_X.append(X)
                all_keys.extend(keys)
            logger.info("推理窗口完成", start=cs, end=ce, read_start=read_start, samples=len(keys))
        if not all_keys:
            raise RuntimeError(f"测试区间无可预测样本: {sd} ~ {ed}")
        X = np.concatenate(all_X, axis=0)
        idx = pd.DataFrame(all_keys, columns=["date", "instrument"])
        idx["date"] = pd.to_datetime(idx["date"]).dt.normalize()
        logger.info("测试集构建完成", samples=len(idx), elapsed=round(time.time() - t0, 2))
        return X, idx

    def fallback_scores(date_values, instrument_values):
        """确定性极小截面 fallback，避免整日常数在平台 z-score 后变成全 NaN。"""
        inst = pd.Series(instrument_values, dtype="string").astype(str)
        dates = pd.Series(date_values, dtype="string").astype(str)
        raw = inst.str.cat(dates, sep="_")
        h = pd.util.hash_pandas_object(raw, index=False).to_numpy(np.uint64).astype(np.float64)
        z = (h / float(np.iinfo(np.uint64).max)) - 0.5
        return z.astype(np.float64)

    def daily_winsor_zscore(df):
        parts = []
        for d, sub in df.groupby("date", sort=False):
            out = sub.copy()
            fallback = fallback_scores(out["date"].astype(str).to_numpy(), out["instrument"].astype(str).to_numpy())
            fallback_series = pd.Series(fallback, index=out.index, dtype="float64")
            s = out["score"].astype("float64").replace([np.inf, -np.inf], np.nan)
            missing = s.isna().to_numpy()
            s = s.fillna(0.001 * fallback_series)
            if len(s) >= 2:
                lo, hi = s.quantile([0.01, 0.99])
                s = s.clip(lo, hi)
                std = s.std(ddof=0)
                if np.isfinite(std) and std > 1e-12:
                    s = (s - s.mean()) / std
                else:
                    # 整日无预测或模型分数常数时，使用确定性截面 fallback，保证不触发整日全缺。
                    s = fallback_series.copy()
                    s = (s - s.mean()) / (s.std(ddof=0) + 1e-12)
            else:
                s = fallback_series.copy()
            # 对缺预测股票使用很小但非相同的截面值；已有预测保留标准化后的模型排序。
            if missing.any() and (~missing).any():
                arr = s.to_numpy(dtype=np.float64, copy=True)
                arr[missing] = 0.001 * fallback[missing]
                s = pd.Series(arr, index=s.index)
            out["score"] = s.to_numpy(np.float64)
            parts.append(out)
        return pd.concat(parts, ignore_index=True)

    # ---------- 加载权重 ----------
    model_path = resolve_model_path()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = load_json_model(model_path, map_location=device)
    stats = (np.asarray(ckpt["mean"], np.float32), np.asarray(ckpt["std"], np.float32))
    model_cfg = ckpt.get(
        "model_cfg",
        {
            "n_feat": N_FEAT,
            "hidden_size": 128,
            "num_layers": 2,
            "dropout": 0.1,
            "rnn_type": "GRU",
        },
    )
    model = StockALSTM(**model_cfg).to(device)
    model.load_state_dict(ckpt["state_dict"], strict=True)
    model.eval()
    logger.info(
        "已加载 StockALSTM 模型",
        path=model_path,
        device=str(device),
        val_metrics=ckpt.get("val_metrics"),
    )

    # ---------- 推理 ----------
    stock_pool = pool(start_date, end_date)
    instruments = stock_pool["instrument"].drop_duplicates().tolist()
    Xte, idx_df = build_infer_dataset(INFER_TABLE, start_date, end_date, instruments, stats)
    preds = []
    with torch.no_grad():
        for i in range(0, len(Xte), PRED_BATCH):
            xb = torch.from_numpy(Xte[i:i + PRED_BATCH]).to(device)
            preds.append(model(xb).detach().cpu().numpy())
    idx_df["score"] = np.concatenate(preds).astype(np.float64)
    idx_df = idx_df.drop_duplicates(["date", "instrument"], keep="last")

    # 左连接官方股票池；缺预测股票先保留 NaN，后续用确定性 fallback 填充，避免整日常数。
    result = stock_pool.merge(idx_df, on=["date", "instrument"], how="left")
    result["score"] = result["score"].replace([np.inf, -np.inf], np.nan)
    result = daily_winsor_zscore(result)
    result = result[["date", "instrument", "score"]].reset_index(drop=True)
    logger.info("分数构建完成", rows=len(result), days=result["date"].nunique(),
                instruments=result["instrument"].nunique())
    return result


In [ ]:
# -*- coding: utf-8 -*-

import json
import pandas as pd

from bigquant import dai


# ===============================
# 配置
# ===============================

LOCAL_JSON = "alstm_model.json"

TABLE = "bigalpha_2026_stock_bar1m"

START = "2022-01-01"
END = "2024-12-31"


# ===============================
# 读取本地top1000
# ===============================

with open(
    LOCAL_JSON,
    "r",
    encoding="utf-8"
) as f:
    local_stocks = json.load(f)


local_stocks = set(
    str(x)
    for x in local_stocks
)


print("="*50)
print("Local top1000")
print("count:", len(local_stocks))


# ===============================
# 读取云端股票池
# ===============================

sql = f"""
SELECT DISTINCT instrument
FROM {TABLE}
ORDER BY instrument
"""


cloud_df = dai.query(
    sql,
    filters={
        "date":[START, END]
    }
).df()


cloud_stocks = set(
    cloud_df["instrument"]
    .astype(str)
)


print("="*50)
print("Cloud stock pool")
print("count:", len(cloud_stocks))


# ===============================
# 对比
# ===============================

intersection = (
    local_stocks
    &
    cloud_stocks
)

only_local = (
    local_stocks
    -
    cloud_stocks
)

only_cloud = (
    cloud_stocks
    -
    local_stocks
)


print("="*50)
print("Intersection")
print(
    len(intersection),
    "/",
    len(local_stocks),
    "=",
    len(intersection)/len(local_stocks)
)


print("="*50)
print("Local but not cloud")
print(
    len(only_local)
)

print(sorted(list(only_local))[:50])


print("="*50)
print("Cloud but not local")
print(
    len(only_cloud)
)

print(sorted(list(only_cloud))[:50])


# ===============================
# 保存差异
# ===============================

with open(
    "local_not_cloud.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        sorted(list(only_local)),
        f,
        ensure_ascii=False,
        indent=2
    )


with open(
    "cloud_not_local.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        sorted(list(only_cloud)),
        f,
        ensure_ascii=False,
        indent=2
    )


print("="*50)
print("saved:")
print("local_not_cloud.json")
print("cloud_not_local.json")

Local top1000
count: 182
Cloud stock pool
count: 2241
Intersection
182 / 182 = 1.0
Local but not cloud
0
[]
Cloud but not local
2059
['000038.SZ', '000040.SZ', '000048.SZ', '000049.SZ', '000056.SZ', '000058.SZ', '000059.SZ', '000062.SZ', '000070.SZ', '000088.SZ', '000089.SZ', '000090.SZ', '000099.SZ', '000150.SZ', '000151.SZ', '000156.SZ', '000301.SZ', '000403.SZ', '000404.SZ', '000415.SZ', '000420.SZ', '000422.SZ', '000426.SZ', '000498.SZ', '000503.SZ', '000506.SZ', '000507.SZ', '000514.SZ', '000516.SZ', '000517.SZ', '000520.SZ', '000525.SZ', '000528.SZ', '000532.SZ', '000533.SZ', '000534.SZ', '000539.SZ', '000540.SZ', '000543.SZ', '000544.SZ', '000545.SZ', '000546.SZ', '000550.SZ', '000552.SZ', '000555.SZ', '000558.SZ', '000560.SZ', '000561.SZ', '000563.SZ', '000571.SZ']
saved:
local_not_cloud.json
cloud_not_local.json


In [2]:
import os

import numpy as np
import pandas as pd
import dai
import torch
import structlog
if __name__ == "__main__":
    from bigmodule import M

    # 只用 1 分钟 K 线作为输入数据
    datasources = {"bar1m": "bigalpha_2026_stock_bar1m"}
    # 本地用一小段区间模拟「平台注入的测试集区间」(训练区间已在 transformer_train.py 内写死)
    start_date, end_date = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())

    # 评估系统: 分数经风格剔除后等价于每日单因子, show=True 画绩效图 (IC / 分组 / 压力期)
    result = M.bigalpha_eval._latest(factor_data=score_data, show=True)

[2026-08-05 08:05:12] [info     ] 已加载 StockALSTM 模型              device=cuda path=alstm_model2.json val_metrics=None


RuntimeError: Query interrupted